In [ ]:
"""
================================================================================
TUTORIAL: DATA ENGINEER SENIOR — PANDAS CON GRANDES VOLÚMENES
ORACLE / AZURE SQL — MEJORES PRÁCTICAS PRODUCTIVAS
================================================================================
Autor  : Data Engineer Senior
Stack  : Python 3.10+ | Pandas | NumPy | SQLAlchemy | oracledb | pyodbc
Enfoque: ETL production-ready, validaciones robustas, logging profesional
================================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS Y CONFIGURACIÓN GLOBAL
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import logging
import time
import gc
import os
import json
from pathlib import Path
from datetime import datetime
from typing import Optional, Iterator

# SQLAlchemy (motor de conexión unificado)
from sqlalchemy import create_engine, text, event
from sqlalchemy.exc import SQLAlchemyError
from sqlalchemy.pool import QueuePool

# ── Logging profesional ──────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("etl_pipeline.log", encoding="utf-8"),
    ],
)
log = logging.getLogger("de_senior")

# ── Pandas — configuración global de rendimiento ────────────────────────────
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)
pd.options.mode.copy_on_write = True          # Pandas 2.x — evita SettingWithCopy


# ─────────────────────────────────────────────────────────────────────────────
# 1. CONEXIONES — ORACLE Y AZURE SQL
# ─────────────────────────────────────────────────────────────────────────────

class DBConnectionFactory:
    """
    Fábrica de conexiones reutilizable.
    Centraliza credenciales, pool y reconexión automática.
    BEST PRACTICE: nunca hardcodear credenciales — usar variables de entorno.
    """

    # ── 1A. Oracle (oracledb — driver nativo, sin cliente Oracle) ───────────
    @staticmethod
    def oracle_engine(
        host: str = None,
        port: int = 1521,
        service: str = None,
        user: str = None,
        password: str = None,
        pool_size: int = 5,
    ):
        """
        Crea engine SQLAlchemy para Oracle Database.
        Lee credenciales de variables de entorno si no se pasan.
        """
        try:
            host     = host     or os.environ["ORA_HOST"]
            service  = service  or os.environ["ORA_SERVICE"]
            user     = user     or os.environ["ORA_USER"]
            password = password or os.environ["ORA_PASSWORD"]

            dsn = f"oracle+oracledb://{user}:{password}@{host}:{port}/?service_name={service}"

            engine = create_engine(
                dsn,
                poolclass=QueuePool,
                pool_size=pool_size,
                max_overflow=10,
                pool_pre_ping=True,        # valida conexión antes de usarla
                pool_recycle=3600,         # recicla conexiones cada hora
                echo=False,
            )

            # Verificar conectividad
            with engine.connect() as conn:
                conn.execute(text("SELECT 1 FROM DUAL"))

            log.info("✅ Oracle engine listo | host=%s service=%s", host, service)
            return engine

        except KeyError as e:
            log.error("❌ Variable de entorno faltante: %s", e)
            raise
        except SQLAlchemyError as e:
            log.error("❌ Error al conectar Oracle: %s", e)
            raise

    # ── 1B. Azure SQL / SQL Server (pyodbc) ─────────────────────────────────
    @staticmethod
    def azure_sql_engine(
        server: str = None,
        database: str = None,
        user: str = None,
        password: str = None,
        driver: str = "ODBC Driver 18 for SQL Server",
        pool_size: int = 5,
    ):
        """
        Crea engine SQLAlchemy para Azure SQL Database / SQL Server.
        Soporta autenticación SQL y Azure AD.
        """
        try:
            server   = server   or os.environ["AZ_SERVER"]
            database = database or os.environ["AZ_DATABASE"]
            user     = user     or os.environ["AZ_USER"]
            password = password or os.environ["AZ_PASSWORD"]

            conn_str = (
                f"mssql+pyodbc://{user}:{password}@{server}/{database}"
                f"?driver={driver.replace(' ', '+')}"
                f"&Encrypt=yes&TrustServerCertificate=no&Connection+Timeout=30"
            )

            engine = create_engine(
                conn_str,
                poolclass=QueuePool,
                pool_size=pool_size,
                max_overflow=10,
                pool_pre_ping=True,
                pool_recycle=3600,
                echo=False,
            )

            with engine.connect() as conn:
                conn.execute(text("SELECT 1"))

            log.info("✅ Azure SQL engine listo | server=%s db=%s", server, database)
            return engine

        except KeyError as e:
            log.error("❌ Variable de entorno faltante: %s", e)
            raise
        except SQLAlchemyError as e:
            log.error("❌ Error al conectar Azure SQL: %s", e)
            raise


# ─────────────────────────────────────────────────────────────────────────────
# 2. EXTRACCIÓN — CHUNKING PARA GRANDES VOLÚMENES
# ─────────────────────────────────────────────────────────────────────────────

class DataExtractor:
    """
    Extracción eficiente con chunking, dtype inference y logging de métricas.
    BEST PRACTICE: nunca cargar millones de filas de un solo jalón.
    """

    CHUNK_SIZE = 100_000   # filas por chunk — ajustar según RAM disponible

    def __init__(self, engine):
        self.engine = engine

    # ── 2A. Lectura completa (datasets medianos < 5M filas) ─────────────────
    def read_table(
        self,
        query: str,
        dtype: Optional[dict] = None,
        parse_dates: Optional[list] = None,
    ) -> pd.DataFrame:
        """
        Lee un query completo con dtypes explícitos para reducir memoria.
        """
        t0 = time.perf_counter()
        try:
            df = pd.read_sql(
                sql=text(query),
                con=self.engine.connect(),
                dtype=dtype,
                parse_dates=parse_dates,
            )
            elapsed = time.perf_counter() - t0
            mem_mb  = df.memory_usage(deep=True).sum() / 1_048_576

            log.info(
                "📥 Extracción completa | filas=%s cols=%s mem=%.1fMB tiempo=%.2fs",
                f"{len(df):,}", df.shape[1], mem_mb, elapsed,
            )
            return df

        except SQLAlchemyError as e:
            log.error("❌ Error en extracción: %s", e)
            raise
        except Exception as e:
            log.error("❌ Error inesperado en read_table: %s", e)
            raise

    # ── 2B. Lectura en chunks (datasets grandes > 5M filas) ─────────────────
    def read_in_chunks(
        self,
        query: str,
        dtype: Optional[dict] = None,
        parse_dates: Optional[list] = None,
    ) -> Iterator[pd.DataFrame]:
        """
        Generador: produce chunks de DataFrame para procesar en batch.
        Permite procesar datasets que no caben en RAM.

        Uso:
            for chunk in extractor.read_in_chunks(sql):
                proceso(chunk)
        """
        try:
            total_rows = 0
            chunk_num  = 0
            t0 = time.perf_counter()

            with self.engine.connect() as conn:
                for chunk in pd.read_sql(
                    sql=text(query),
                    con=conn,
                    chunksize=self.CHUNK_SIZE,
                    dtype=dtype,
                    parse_dates=parse_dates,
                ):
                    chunk_num  += 1
                    total_rows += len(chunk)
                    log.info("  📦 Chunk %d | filas_acum=%s", chunk_num, f"{total_rows:,}")
                    yield chunk

            elapsed = time.perf_counter() - t0
            log.info(
                "✅ Chunking completo | total_filas=%s chunks=%d tiempo=%.2fs",
                f"{total_rows:,}", chunk_num, elapsed,
            )

        except SQLAlchemyError as e:
            log.error("❌ Error en read_in_chunks: %s", e)
            raise

    # ── 2C. Lectura desde archivos planos (CSV / JSON / TXT) ────────────────
    @staticmethod
    def read_flat_file(
        filepath: str | Path,
        file_type: str = "csv",
        dtype: Optional[dict] = None,
        parse_dates: Optional[list] = None,
        encoding: str = "utf-8",
        sep: str = ",",
    ) -> pd.DataFrame:
        """
        Lectura robusta de archivos planos con validación de existencia y tipo.
        """
        path = Path(filepath)
        try:
            if not path.exists():
                raise FileNotFoundError(f"Archivo no encontrado: {path}")

            t0 = time.perf_counter()

            if file_type == "csv":
                df = pd.read_csv(
                    path,
                    dtype=dtype,
                    parse_dates=parse_dates,
                    encoding=encoding,
                    sep=sep,
                    low_memory=False,       # evita dtype mixto silencioso
                    on_bad_lines="warn",    # loguea líneas malformadas
                )
            elif file_type == "json":
                df = pd.read_json(path, dtype=dtype, encoding=encoding)
            elif file_type in ("txt", "pipe"):
                df = pd.read_csv(
                    path, sep="|", dtype=dtype,
                    encoding=encoding, low_memory=False,
                )
            elif file_type == "parquet":
                df = pd.read_parquet(path)           # ya tiene dtypes embebidos
            else:
                raise ValueError(f"Tipo de archivo no soportado: {file_type}")

            elapsed = time.perf_counter() - t0
            log.info(
                "📄 Archivo leído: %s | filas=%s cols=%s tiempo=%.2fs",
                path.name, f"{len(df):,}", df.shape[1], elapsed,
            )
            return df

        except (FileNotFoundError, ValueError) as e:
            log.error("❌ %s", e)
            raise
        except Exception as e:
            log.error("❌ Error leyendo archivo %s: %s", filepath, e)
            raise


# ─────────────────────────────────────────────────────────────────────────────
# 3. OPTIMIZACIÓN DE MEMORIA — TÉCNICA ESENCIAL PARA GRANDES VOLÚMENES
# ─────────────────────────────────────────────────────────────────────────────

class MemoryOptimizer:
    """
    Reduce hasta 70-80% el uso de RAM ajustando dtypes automáticamente.
    BEST PRACTICE obligatoria antes de transformar datasets grandes.
    """

    @staticmethod
    def optimize(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
        """
        Downcasting inteligente por tipo de columna:
          - int64  → int8/16/32 según rango de valores
          - float64 → float32
          - object con baja cardinalidad → category
        """
        try:
            mem_before = df.memory_usage(deep=True).sum() / 1_048_576

            df_opt = df.copy()

            for col in df_opt.columns:
                col_type = df_opt[col].dtype

                # Enteros
                if pd.api.types.is_integer_dtype(col_type):
                    df_opt[col] = pd.to_numeric(df_opt[col], downcast="integer")

                # Flotantes
                elif pd.api.types.is_float_dtype(col_type):
                    df_opt[col] = pd.to_numeric(df_opt[col], downcast="float")

                # Texto → category si cardinalidad < 50% del total
                elif col_type == object:
                    num_unique = df_opt[col].nunique()
                    if num_unique / len(df_opt) < 0.50:
                        df_opt[col] = df_opt[col].astype("category")

            mem_after = df_opt.memory_usage(deep=True).sum() / 1_048_576
            reduccion  = (1 - mem_after / mem_before) * 100

            if verbose:
                log.info(
                    "🔧 Memoria optimizada | antes=%.1fMB después=%.1fMB reducción=%.1f%%",
                    mem_before, mem_after, reduccion,
                )
            return df_opt

        except Exception as e:
            log.error("❌ Error en optimize_memory: %s", e)
            raise

    @staticmethod
    def memory_report(df: pd.DataFrame) -> pd.DataFrame:
        """Reporte detallado de consumo por columna."""
        try:
            report = pd.DataFrame({
                "dtype"   : df.dtypes,
                "nulls"   : df.isnull().sum(),
                "nulls_%"  : (df.isnull().mean() * 100).round(2),
                "unique"  : df.nunique(),
                "mem_MB"  : df.memory_usage(deep=True)[1:] / 1_048_576,
            })
            report = report.sort_values("mem_MB", ascending=False)
            return report
        except Exception as e:
            log.error("❌ Error en memory_report: %s", e)
            raise


# ─────────────────────────────────────────────────────────────────────────────
# 4. TRANSFORMACIONES — VECTORIZACIÓN VS LOOPS (ANTIPATRÓN)
# ─────────────────────────────────────────────────────────────────────────────

class DataTransformer:
    """
    Transformaciones ETL production-ready con vectorización.
    REGLA DE ORO: nunca usar .iterrows() en producción con datos grandes.
    """

    # ── 4A. Limpieza base ────────────────────────────────────────────────────
    @staticmethod
    def clean_strings(df: pd.DataFrame, cols: list) -> pd.DataFrame:
        """Strip + upper + reemplaza espacios múltiples — vectorizado."""
        try:
            for col in cols:
                if col in df.columns:
                    df[col] = (
                        df[col]
                        .astype(str)
                        .str.strip()
                        .str.upper()
                        .str.replace(r"\s+", " ", regex=True)
                        .replace("NAN", np.nan)
                    )
            return df
        except Exception as e:
            log.error("❌ clean_strings | col=%s err=%s", col, e)
            raise

    # ── 4B. Transformación con np.where / np.select (rápido) ────────────────
    @staticmethod
    def categorize_amount(df: pd.DataFrame, amount_col: str) -> pd.DataFrame:
        """
        Clasifica montos en segmentos usando np.select (vectorizado).
        10-100x más rápido que apply() o iterrows().
        """
        try:
            conditions = [
                df[amount_col] < 1_000,
                df[amount_col].between(1_000, 9_999.99),
                df[amount_col].between(10_000, 99_999.99),
                df[amount_col] >= 100_000,
            ]
            choices = ["MICRO", "PEQUEÑO", "MEDIANO", "GRANDE"]

            df["segmento_monto"] = np.select(
                conditions, choices, default="SIN_CLASIFICAR"
            )
            return df
        except KeyError:
            log.error("❌ Columna '%s' no encontrada", amount_col)
            raise
        except Exception as e:
            log.error("❌ categorize_amount: %s", e)
            raise

    # ── 4C. Fechas — manejo robusto ──────────────────────────────────────────
    @staticmethod
    def parse_dates_safe(
        df: pd.DataFrame,
        date_cols: list,
        fmt: str = "%Y-%m-%d",
    ) -> pd.DataFrame:
        """
        Parsea fechas con errors='coerce' (nunca rompe el pipeline).
        Agrega columnas derivadas: año, mes, trimestre, día_semana.
        """
        try:
            for col in date_cols:
                if col not in df.columns:
                    log.warning("⚠️  Columna fecha '%s' no existe — omitida", col)
                    continue

                df[col] = pd.to_datetime(df[col], format=fmt, errors="coerce")

                nulls = df[col].isnull().sum()
                if nulls > 0:
                    log.warning(
                        "⚠️  '%s' tiene %s fechas inválidas (→ NaT)", col, f"{nulls:,}"
                    )

                # Columnas derivadas útiles para analítica dimensional
                df[f"{col}_anio"]       = df[col].dt.year.astype("Int16")
                df[f"{col}_mes"]        = df[col].dt.month.astype("Int8")
                df[f"{col}_trim"]       = df[col].dt.quarter.astype("Int8")
                df[f"{col}_dia_semana"] = df[col].dt.day_name()

            return df
        except Exception as e:
            log.error("❌ parse_dates_safe: %s", e)
            raise

    # ── 4D. ANTI-PATRÓN vs BEST PRACTICE (ejemplo didáctico) ────────────────
    @staticmethod
    def demo_vectorization(df: pd.DataFrame, col: str) -> None:
        """Muestra diferencia de velocidad entre iterrows y vectorización."""

        n = min(len(df), 200_000)
        sample = df[[col]].head(n).copy()

        # ❌ ANTI-PATRÓN — nunca en producción
        t0 = time.perf_counter()
        result_slow = []
        for _, row in sample.iterrows():
            result_slow.append(row[col] * 1.16)
        slow_time = time.perf_counter() - t0

        # ✅ BEST PRACTICE — vectorizado
        t0 = time.perf_counter()
        result_fast = sample[col] * 1.16
        fast_time = time.perf_counter() - t0

        speedup = slow_time / fast_time if fast_time > 0 else float("inf")
        log.info(
            "⚡ Vectorización vs iterrows | lento=%.3fs rápido=%.4fs speedup=%.0fx",
            slow_time, fast_time, speedup,
        )


# ─────────────────────────────────────────────────────────────────────────────
# 5. CALIDAD DE DATOS (DQ) — VALIDACIONES OBLIGATORIAS EN PRODUCCIÓN
# ─────────────────────────────────────────────────────────────────────────────

class DataQualityChecker:
    """
    Suite de validaciones DQ para pipelines productivos.
    Genera reporte estructurado con severidad por regla.
    """

    def __init__(self, df: pd.DataFrame, dataset_name: str = "dataset"):
        self.df   = df
        self.name = dataset_name
        self.issues: list[dict] = []

    def _add_issue(self, rule: str, severity: str, detail: str, count: int = 0):
        self.issues.append({
            "dataset"  : self.name,
            "regla"    : rule,
            "severidad": severity,        # CRITICAL / WARNING / INFO
            "detalle"  : detail,
            "conteo"   : count,
            "ts"       : datetime.now().isoformat(),
        })

    # ── 5A. Nulos ────────────────────────────────────────────────────────────
    def check_nulls(self, critical_cols: list, warn_threshold: float = 0.05):
        """
        critical_cols: cualquier nulo es CRITICAL.
        Resto de columnas: WARNING si > warn_threshold (default 5%).
        """
        try:
            for col in self.df.columns:
                null_pct = self.df[col].isnull().mean()
                null_cnt = self.df[col].isnull().sum()

                if col in critical_cols and null_cnt > 0:
                    self._add_issue(
                        "NULL_CRITICO", "CRITICAL",
                        f"Columna clave '{col}' tiene {null_cnt:,} nulos ({null_pct:.1%})",
                        null_cnt,
                    )
                elif null_pct > warn_threshold:
                    self._add_issue(
                        "NULL_ALTO", "WARNING",
                        f"'{col}' supera umbral de nulos: {null_pct:.1%}",
                        null_cnt,
                    )
        except Exception as e:
            log.error("❌ check_nulls: %s", e)
            raise

    # ── 5B. Duplicados ───────────────────────────────────────────────────────
    def check_duplicates(self, key_cols: list):
        """Valida unicidad de llave de negocio."""
        try:
            dupes = self.df.duplicated(subset=key_cols).sum()
            if dupes > 0:
                self._add_issue(
                    "DUPLICADOS", "CRITICAL",
                    f"Llave {key_cols} tiene {dupes:,} duplicados",
                    dupes,
                )
            else:
                self._add_issue("DUPLICADOS", "INFO", "Sin duplicados en llave de negocio")
        except KeyError as e:
            log.error("❌ Columna de llave no encontrada: %s", e)
            raise

    # ── 5C. Rangos numéricos ─────────────────────────────────────────────────
    def check_range(self, col: str, min_val: float, max_val: float):
        """Detecta outliers fuera de rango de negocio esperado."""
        try:
            if col not in self.df.columns:
                log.warning("⚠️  '%s' no existe para check_range", col)
                return

            out_of_range = ((self.df[col] < min_val) | (self.df[col] > max_val)).sum()
            if out_of_range > 0:
                self._add_issue(
                    "FUERA_DE_RANGO", "WARNING",
                    f"'{col}' tiene {out_of_range:,} valores fuera de [{min_val}, {max_val}]",
                    out_of_range,
                )
        except Exception as e:
            log.error("❌ check_range '%s': %s", col, e)
            raise

    # ── 5D. Reporte final ────────────────────────────────────────────────────
    def report(self, fail_on_critical: bool = True) -> pd.DataFrame:
        """
        Retorna DataFrame con todos los issues.
        fail_on_critical=True: lanza excepción si hay CRITICALs (recomendado en prod).
        """
        try:
            if not self.issues:
                log.info("✅ DQ: Sin issues encontrados en '%s'", self.name)
                return pd.DataFrame()

            df_report = pd.DataFrame(self.issues)

            criticals = df_report[df_report["severidad"] == "CRITICAL"]
            warnings  = df_report[df_report["severidad"] == "WARNING"]

            log.info(
                "📋 DQ Report '%s' | CRITICALs=%d WARNINGs=%d",
                self.name, len(criticals), len(warnings),
            )

            if fail_on_critical and not criticals.empty:
                raise ValueError(
                    f"Pipeline detenido: {len(criticals)} issues CRITICAL en '{self.name}'"
                )

            return df_report

        except ValueError:
            raise
        except Exception as e:
            log.error("❌ DQ report: %s", e)
            raise


# ─────────────────────────────────────────────────────────────────────────────
# 6. CARGA (LOAD) — UPSERT EFICIENTE HACIA DWH
# ─────────────────────────────────────────────────────────────────────────────

class DataLoader:
    """
    Carga hacia Oracle / Azure SQL con:
      - Carga en chunks (evita timeouts y OOM)
      - Estrategias: replace / append / upsert
      - Logging de métricas de carga
    """

    LOAD_CHUNK = 10_000    # filas por batch insert

    def __init__(self, engine):
        self.engine = engine

    # ── 6A. Carga simple (replace / append) ──────────────────────────────────
    def load_table(
        self,
        df: pd.DataFrame,
        table_name: str,
        schema: str = None,
        if_exists: str = "append",   # replace | append | fail
    ) -> int:
        """
        Carga DataFrame → base de datos en chunks.
        Retorna número de filas insertadas.
        """
        try:
            t0    = time.perf_counter()
            rows  = len(df)

            df.to_sql(
                name       = table_name,
                con        = self.engine,
                schema     = schema,
                if_exists  = if_exists,
                index      = False,
                chunksize  = self.LOAD_CHUNK,
                method     = "multi",       # inserts en batch (más rápido)
            )

            elapsed = time.perf_counter() - t0
            log.info(
                "💾 Carga OK | tabla=%s.%s filas=%s tiempo=%.2fs throughput=%.0f fila/s",
                schema or "default", table_name,
                f"{rows:,}", elapsed, rows / elapsed,
            )
            return rows

        except SQLAlchemyError as e:
            log.error("❌ Error al cargar tabla '%s': %s", table_name, e)
            raise
        except Exception as e:
            log.error("❌ Error inesperado en load_table: %s", e)
            raise

    # ── 6B. Upsert manual (MERGE / INSERT ON CONFLICT) ───────────────────────
    def upsert_oracle(
        self,
        df: pd.DataFrame,
        stage_table: str,
        target_table: str,
        key_cols: list,
        schema: str = None,
    ) -> None:
        """
        Patrón MERGE de Oracle:
          1. Carga datos en tabla staging (truncate + insert)
          2. Ejecuta MERGE desde staging → target
        Evita duplicados y maneja actualizaciones en un solo paso.
        """
        try:
            full_stage  = f"{schema}.{stage_table}"  if schema else stage_table
            full_target = f"{schema}.{target_table}" if schema else target_table

            # Paso 1: truncar staging y cargar
            with self.engine.begin() as conn:
                conn.execute(text(f"TRUNCATE TABLE {full_stage}"))
            self.load_table(df, stage_table, schema=schema, if_exists="append")

            # Paso 2: construir MERGE dinámico
            non_key_cols = [c for c in df.columns if c not in key_cols]
            on_clause    = " AND ".join([f"t.{k} = s.{k}" for k in key_cols])
            update_set   = ", ".join([f"t.{c} = s.{c}" for c in non_key_cols])
            insert_cols  = ", ".join(df.columns)
            insert_vals  = ", ".join([f"s.{c}" for c in df.columns])

            merge_sql = f"""
                MERGE INTO {full_target} t
                USING {full_stage} s
                ON ({on_clause})
                WHEN MATCHED THEN
                    UPDATE SET {update_set}
                WHEN NOT MATCHED THEN
                    INSERT ({insert_cols})
                    VALUES ({insert_vals})
            """

            with self.engine.begin() as conn:
                conn.execute(text(merge_sql))

            log.info("🔄 MERGE Oracle OK | stage=%s → target=%s", full_stage, full_target)

        except SQLAlchemyError as e:
            log.error("❌ Upsert Oracle error: %s", e)
            raise


# ─────────────────────────────────────────────────────────────────────────────
# 7. PIPELINE ETL COMPLETO — ORQUESTACIÓN
# ─────────────────────────────────────────────────────────────────────────────

def run_etl_pipeline(
    source_engine,
    target_engine,
    source_query: str,
    target_table: str,
    key_cols: list,
    schema: str = None,
    use_chunks: bool = True,
) -> dict:
    """
    Pipeline ETL end-to-end production-ready:
      Extract → Optimize → Quality Check → Transform → Load

    Retorna dict con métricas del pipeline.
    """
    metrics = {
        "pipeline_start" : datetime.now().isoformat(),
        "status"         : "RUNNING",
        "rows_extracted" : 0,
        "rows_loaded"    : 0,
        "dq_issues"      : 0,
        "error"          : None,
    }

    t_global = time.perf_counter()

    try:
        extractor   = DataExtractor(source_engine)
        loader      = DataLoader(target_engine)
        transformer = DataTransformer()
        optimizer   = MemoryOptimizer()

        # ── EXTRACT ──────────────────────────────────────────────────────────
        log.info("═" * 60)
        log.info("🚀 Iniciando ETL → tabla: %s", target_table)

        if use_chunks:
            frames = []
            for chunk in extractor.read_in_chunks(source_query):
                chunk = optimizer.optimize(chunk, verbose=False)
                frames.append(chunk)
            df = pd.concat(frames, ignore_index=True)
            del frames; gc.collect()
        else:
            df = extractor.read_table(source_query)
            df = optimizer.optimize(df)

        metrics["rows_extracted"] = len(df)

        # ── QUALITY CHECK ────────────────────────────────────────────────────
        dq = DataQualityChecker(df, dataset_name=target_table)
        dq.check_nulls(critical_cols=key_cols)
        dq.check_duplicates(key_cols=key_cols)
        dq_report = dq.report(fail_on_critical=True)
        metrics["dq_issues"] = len(dq_report)

        # ── TRANSFORM ────────────────────────────────────────────────────────
        # (aquí van las reglas de negocio específicas del pipeline)
        df["ETL_LOAD_TS"] = datetime.now()

        # ── LOAD ─────────────────────────────────────────────────────────────
        rows_loaded = loader.load_table(df, target_table, schema=schema)
        metrics["rows_loaded"] = rows_loaded
        metrics["status"]      = "SUCCESS"

    except ValueError as e:
        # Issues de calidad críticos
        metrics["status"] = "FAILED_DQ"
        metrics["error"]  = str(e)
        log.error("🛑 Pipeline detenido por DQ: %s", e)
        raise

    except Exception as e:
        metrics["status"] = "FAILED"
        metrics["error"]  = str(e)
        log.error("🛑 Pipeline fallido: %s", e)
        raise

    finally:
        elapsed = time.perf_counter() - t_global
        metrics["duration_sec"]   = round(elapsed, 2)
        metrics["pipeline_end"]   = datetime.now().isoformat()

        # Guardar métricas en JSON para auditoría
        metrics_path = Path(f"metrics_{target_table}_{datetime.now():%Y%m%d_%H%M%S}.json")
        try:
            metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
            log.info("📊 Métricas guardadas: %s", metrics_path)
        except Exception:
            pass

        log.info(
            "🏁 Pipeline '%s' | status=%s filas=%s tiempo=%.2fs",
            target_table, metrics["status"],
            f"{metrics['rows_loaded']:,}", elapsed,
        )

    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# 8. DEMO LOCAL — EJECUTABLE SIN BD (datos sintéticos)
# ─────────────────────────────────────────────────────────────────────────────

def demo_sin_bd():
    """
    Demo completa sin conexión real a BD.
    Genera datos sintéticos para probar todo el stack.
    """
    log.info("=" * 65)
    log.info("🎓 DEMO DATA ENGINEER SENIOR — PANDAS LARGE VOLUMES")
    log.info("=" * 65)

    np.random.seed(42)
    N = 500_000   # 500k filas para simular volumen real

    # ── Generar dataset sintético ────────────────────────────────────────────
    log.info("🔨 Generando %s filas sintéticas...", f"{N:,}")
    try:
        df = pd.DataFrame({
            "ID_TRANSACCION" : np.arange(1, N + 1),
            "ID_CLIENTE"     : np.random.randint(1, 50_001, N),
            "FECHA_TX"       : pd.date_range("2022-01-01", periods=N, freq="1min"),
            "MONTO"          : np.random.exponential(scale=5_000, size=N).round(2),
            "CANAL"          : np.random.choice(["DIGITAL","SUCURSAL","ATM","APP"], N),
            "ESTATUS"        : np.random.choice(["APROBADO","RECHAZADO","PENDIENTE"], N,
                                                 p=[0.85, 0.10, 0.05]),
            "MONEDA"         : np.random.choice(["MXN","USD","EUR"], N, p=[0.80,0.15,0.05]),
            "DESCRIPCION"    : np.random.choice(
                                ["pago servicios","  compra  TIENDA","RETIRO cajero",
                                 "pago TARJETA", "transferencia"], N),
        })

        # Introducir nulos artificiales para DQ
        df.loc[np.random.choice(N, 500, replace=False), "CANAL"]   = np.nan
        df.loc[np.random.choice(N, 200, replace=False), "MONEDA"]  = np.nan
        df.loc[np.random.choice(N, 100, replace=False), "MONTO"]   = -99.99  # datos sucios

    except Exception as e:
        log.error("❌ Error generando datos sintéticos: %s", e)
        raise

    # ── Paso 1: Optimizar memoria ────────────────────────────────────────────
    log.info("\n📌 PASO 1: Optimización de memoria")
    try:
        optimizer = MemoryOptimizer()
        df_opt    = optimizer.optimize(df)
        report    = optimizer.memory_report(df_opt)
        log.info("\n%s", report.to_string())
    except Exception as e:
        log.error("❌ Error en optimización: %s", e)
        raise

    # ── Paso 2: Transformaciones ─────────────────────────────────────────────
    log.info("\n📌 PASO 2: Transformaciones ETL")
    try:
        tx = DataTransformer()
        df_opt = tx.clean_strings(df_opt, ["CANAL", "ESTATUS", "MONEDA", "DESCRIPCION"])
        df_opt = tx.categorize_amount(df_opt, "MONTO")
        df_opt = tx.parse_dates_safe(df_opt, ["FECHA_TX"], fmt="%Y-%m-%d %H:%M:%S")

        log.info("  → Segmentos creados:\n%s",
                 df_opt["segmento_monto"].value_counts().to_string())
    except Exception as e:
        log.error("❌ Error en transformaciones: %s", e)
        raise

    # ── Paso 3: Velocidad vectorización ─────────────────────────────────────
    log.info("\n📌 PASO 3: Demo vectorización")
    try:
        tx.demo_vectorization(df_opt, "MONTO")
    except Exception as e:
        log.warning("⚠️  Demo vectorización omitida: %s", e)

    # ── Paso 4: Data Quality ─────────────────────────────────────────────────
    log.info("\n📌 PASO 4: Validaciones DQ")
    try:
        dq = DataQualityChecker(df_opt, dataset_name="TX_FINANCIERAS")
        dq.check_nulls(critical_cols=["ID_TRANSACCION", "ID_CLIENTE"], warn_threshold=0.01)
        dq.check_duplicates(key_cols=["ID_TRANSACCION"])
        dq.check_range("MONTO", min_val=0, max_val=1_000_000)

        dq_report = dq.report(fail_on_critical=False)   # False para que la demo continúe
        if not dq_report.empty:
            log.info("\n%s", dq_report[["regla","severidad","detalle","conteo"]].to_string(index=False))
    except Exception as e:
        log.error("❌ Error en DQ: %s", e)
        raise

    # ── Paso 5: Agregaciones analíticas ─────────────────────────────────────
    log.info("\n📌 PASO 5: Agregaciones para DWH")
    try:
        summary = (
            df_opt
            .groupby(["CANAL", "segmento_monto", "ESTATUS"], observed=True)
            .agg(
                total_tx    = ("ID_TRANSACCION", "count"),
                monto_total = ("MONTO",          "sum"),
                monto_prom  = ("MONTO",          "mean"),
                monto_max   = ("MONTO",          "max"),
            )
            .round(2)
            .reset_index()
            .sort_values("monto_total", ascending=False)
        )

        log.info("\n%s", summary.head(10).to_string(index=False))
    except Exception as e:
        log.error("❌ Error en agregaciones: %s", e)
        raise

    # ── Paso 6: Exportar a Parquet (formato óptimo para Data Lake) ───────────
    log.info("\n📌 PASO 6: Exportar a Parquet (Data Lake ready)")
    try:
        out_path = Path("/home/claude/tx_financieras_processed.parquet")
        df_opt.to_parquet(out_path, index=False, compression="snappy")
        size_mb = out_path.stat().st_size / 1_048_576
        log.info("  ✅ Parquet exportado: %s | %.1f MB", out_path.name, size_mb)
    except Exception as e:
        log.warning("⚠️  Export Parquet omitido (pyarrow no disponible): %s", e)

    log.info("\n" + "=" * 65)
    log.info("✅ DEMO COMPLETADA EXITOSAMENTE")
    log.info("=" * 65)

    return df_opt, summary


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    try:
        df_result, df_summary = demo_sin_bd()
    except Exception as e:
        log.critical("💥 Error fatal en pipeline: %s", e)
        raise